# Pandas — Reshaping Data

> **Repo:** Python_Libraries | **Library:** Pandas  
> **Notebook:** 08_Reshaping_Data  

---

In [3]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

print("Pandas version:", pd.__version__)

Pandas version: 2.2.2


In [4]:
np.random.seed(42)

months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
          'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
regions = ['North', 'South', 'East', 'West']
products = ['Laptop', 'Tablet', 'Phone', 'Monitor']

rows = []
for month in months:
    for region in regions:
        for product in products:
            rows.append({
                'Month': month,
                'Region': region,
                'Product': product,
                'Units_Sold': np.random.randint(50, 300),
                'Revenue': np.random.randint(10000, 80000)
            })

df = pd.DataFrame(rows)
print(df.shape)
df.sample(10)

(192, 5)


,Month,Region,Product,Units_Sold,Revenue
111,Jul,West,Monitor,236,61663
26,Feb,East,Phone,213,77121
76,May,West,Laptop,177,36105
78,May,West,Phone,266,54262
182,Dec,South,Phone,62,12368
170,Nov,East,Phone,272,33093
10,Jan,East,Phone,107,38693
131,Sep,North,Monitor,201,69040
133,Sep,South,Tablet,124,42711
86,Jun,South,Phone,274,23545


## Section 1: `pivot()` -- Reshape Long to Wide

`pivot()` takes a long-format DataFrame and spreads one column's values across new columns.

**Syntax:** `df.pivot(index=..., columns=..., values=...)`

- `index` → what becomes the row labels
- `columns` → whose unique values become new column headers  
- `values` → what fills the cells

⚠️ Requires unique index/column combinations — no duplicates allowed.

### 1.1 `pivot()` on a subset

In [8]:
# Filter to one region first — pivot needs unique index/column pairs
df_north = df[df['Region'] == 'North'].copy()

# Pivot: rows = Month, columns = Product, values = Units_Sold
pivot_units = df_north.pivot(index='Month', columns='Product', values='Units_Sold')

print("Shape:", pivot_units.shape)
print("\nColumn name after pivot:", pivot_units.columns.name)
pivot_units

Shape: (12, 4)

Column name after pivot: Product


Product,Laptop,Monitor,Phone,Tablet
Month,,,,
Apr,185,280,54,212
Aug,142,145,196,236
Dec,179,219,210,66
Feb,100,278,113,104
Jan,152,252,171,142
Jul,51,196,274,278
Jun,178,52,165,239
Mar,97,239,264,249
May,280,192,278,145


### 1.2 Cleaning up the pivot output

In [10]:
# columns.name='Product' is a leftover label from pivot — clean it
pivot_units.columns.name = None

# Reorder months chronologically
month_order = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']
pivot_units = pivot_units.reindex(month_order)

print("North Region — Monthly Units Sold per Product\n")
pivot_units

North Region — Monthly Units Sold per Product



,Laptop,Monitor,Phone,Tablet
Month,,,,
Jan,152,252,171,142
Feb,100,278,113,104
Mar,97,239,264,249
Apr,185,280,54,212
May,280,192,278,145
Jun,178,52,165,239
Jul,51,196,274,278
Aug,142,145,196,236
Sep,282,201,180,97


In [11]:
# 💡 Interview insight: pivot() breaks if index+column pair isn't unique
# Demonstrate the error:
try:
    df.pivot(index='Month', columns='Product', values='Units_Sold')
except ValueError as e:
    print("❌ ValueError:", e)
    print("\n✅ Fix: Use pivot_table() which handles duplicates via aggregation")

❌ ValueError: Index contains duplicate entries, cannot reshape

✅ Fix: Use pivot_table() which handles duplicates via aggregation


## Section 2: pivot_table() -- Pivot with Aggregation

`pivot_table()` is the powerful version of `pivot()`.

It handles duplicate index/column combinations by aggregating them.

**Syntax:**
`df.pivot_table(index=..., columns=..., values=..., aggfunc=...)`

**Key parameters**:
- `aggfunc` → how to combine duplicates: 'mean', 'sum', 'count', 'max', etc.
- `fill_value` → what to put in empty cells (instead of NaN)
- `margins` → adds row/column totals (like Excel's Grand Total)

### 2.1 Basic Pivot Table

In [20]:
# Full dataset now — pivot_table handles all 4 regions per Month+Product
pt_units = df.pivot_table(
    index='Month',
    columns='Product',
    values='Units_Sold',
    aggfunc='mean'          # average units across all 4 regions
)

pt_units = pt_units.reindex(month_order).round(1)
pt_units.columns.name = None

print("Avg Units Sold per Product per Month (all regions)\n")
pt_units

Avg Units Sold per Product per Month (all regions)



,Laptop,Monitor,Phone,Tablet
Month,,,,
Jan,182.8,228.2,159.2,208.0
Feb,178.8,196.0,156.0,100.5
Mar,167.2,203.2,179.0,221.0
Apr,109.0,221.5,80.8,212.5
May,203.2,195.2,198.0,159.5
Jun,229.5,133.5,189.0,157.8
Jul,186.2,174.0,186.8,230.2
Aug,144.8,225.8,152.0,208.0
Sep,193.8,148.2,193.2,174.8


### 2.2 Pivot Table with Important Parameters

In [24]:
pt_revenue = df.pivot_table(
    index='Region',
    columns='Product',
    values='Revenue',
    aggfunc='sum',
    fill_value=0,
    margins=True,           # adds 'All' row and column
    margins_name='Total'
)

pt_revenue.columns.name = None
print("Total Revenue by Region & Product\n")
pt_revenue

Total Revenue by Region & Product



,Laptop,Monitor,Phone,Tablet,Total
Region,,,,,
East,577335,477226,587292,658724,2300577
North,546795,631743,517835,565620,2261993
South,450263,555090,521010,427835,1954198
West,483730,509400,664921,427911,2085962
Total,2058123,2173459,2291058,2080090,8602730


### 2.3 Multiple Aggregations at once

In [31]:
# pivot_table can compute multiple aggregations simultaneously
pt_multi = df.pivot_table(
    index='Region',
    columns='Product',
    values='Units_Sold',
    aggfunc=['mean', 'max', 'min']
).round(1)

pt_multi.columns.name = None
print("Units Sold — Mean / Max / Min by Region & Product\n")
pt_multi

Units Sold — Mean / Max / Min by Region & Product



mean                          max                         min                     
Product Laptop Monitor  Phone Tablet Laptop Monitor Phone Tablet Laptop Monitor Phone Tablet
Region                                                                                      
East     208.5   189.0  148.8  162.4    294     285   272    275     76     107    79     57
North    184.4   197.1  193.5  167.3    297     280   278    278     51      52    54     55
South    163.4   160.8  157.2  183.2    263     269   274    286     62      57    58     71
West     161.2   208.8  171.1  192.9    263     296   266    296     50      82    53     61

## Section 3: `melt()` -- Reshapes wide to long

`melt()` is the inverse of `pivot()`.

It takes wide-format data (many columns) and collapses it into long format.

**Syntax:**
`df.melt(id_vars=..., value_vars=..., var_name=..., value_name=...)`

**Key parameters:**
- `id_vars`    → columns to keep as-is (identifier columns)
- `value_vars` → columns to collapse (if omitted, all non-id columns are melted)
- `var_name`   → name for the new column holding old column headers
- `value_name` → name for the new column holding the values

**Real-world use:**
- Unpivoting Excel reports
- preparing data for seaborn/plotly
- normalizing survey responses
- database ingestion.

In [35]:
# Start from our cleaned pivot (wide format)
print("Wide format (before melt):")
print(pivot_units.shape, "\n")
pivot_units.head(3)

Wide format (before melt):
(12, 4) 



,Laptop,Monitor,Phone,Tablet
Month,,,,
Jan,152,252,171,142
Feb,100,278,113,104
Mar,97,239,264,249


In [39]:
# Reset index so 'Month' becomes a regular column (id_var)
df_melted = pivot_units.reset_index().melt(
    id_vars='Month',
    value_vars=['Laptop', 'Monitor', 'Phone', 'Tablet'],
    var_name='Product',
    value_name='Units_Sold'
)

df_melted['Month'] = pd.Categorical(df_melted['Month'], categories=month_order, ordered=True)
df_melted = df_melted.sort_values(['Month', 'Product']).reset_index(drop=True)

print("Long format (after melt):")
print(df_melted.shape, "\n")
df_melted.head(12)

Long format (after melt):
(48, 3) 



,Month,Product,Units_Sold
0,Jan,Laptop,152
1,Feb,Laptop,100
2,Mar,Laptop,97
3,Apr,Laptop,185
4,May,Laptop,280
5,Jun,Laptop,178
6,Jul,Laptop,51
7,Aug,Laptop,142
8,Sep,Laptop,282
9,Oct,Laptop,297


In [41]:
# Common scenario: dataset has mixed columns — only melt the metric columns
df_wide_sample = df.pivot_table(
    index=['Month', 'Region'],
    columns='Product',
    values='Revenue',
    aggfunc='sum'
).reset_index()

df_wide_sample.columns.name = None
print("Wide sample (Month + Region as identifiers, Products as value columns):")
print(df_wide_sample.shape)
df_wide_sample.head(6)

Wide sample (Month + Region as identifiers, Products as value columns):
(48, 6)


,Month,Region,Laptop,Monitor,Phone,Tablet
0,Apr,East,73335,37266,18110,34538
1,Apr,North,23986,74505,13561,22666
2,Apr,South,62251,45222,62256,18392
3,Apr,West,16910,60015,33419,10206
4,Aug,East,57254,26371,26646,70713
5,Aug,North,59811,77270,21411,44754


In [43]:
df_long_again = df_wide_sample.melt(
    id_vars=['Month', 'Region'],       # keep these as identifiers
    value_vars=['Laptop', 'Monitor', 'Phone', 'Tablet'],
    var_name='Product',
    value_name='Revenue'
)

df_long_again['Month'] = pd.Categorical(df_long_again['Month'], categories=month_order, ordered=True)
df_long_again = df_long_again.sort_values(['Month', 'Region', 'Product']).reset_index(drop=True)

print("Fully restored long format:")
print(df_long_again.shape)
df_long_again.head(12)

Fully restored long format:
(192, 4)


,Month,Region,Product,Revenue
0,Jan,East,Laptop,77969
1,Jan,East,Monitor,35658
2,Jan,East,Phone,38693
3,Jan,East,Tablet,63707
4,Jan,North,Laptop,25795
5,Jan,North,Monitor,54131
6,Jan,North,Phone,47194
7,Jan,North,Tablet,64886
8,Jan,South,Laptop,26023
9,Jan,South,Monitor,72955


## Section 4: `stack() / unstack()` -- MultiIndex Reshaping

`stack()` and `unstack()` reshape by moving levels between columns and index.

- `stack()`   → moves column headers **into** the row index (wide → long, index-based)
- `unstack()` → moves a row index level **into** columns (long → wide, index-based)

**Key difference from pivot/melt:**

pivot/melt work on regular columns.

stack/unstack work on the Index itself — essential for MultiIndex DataFrames.

**Syntax:**
`df.stack(level=-1)`    # -1 = innermost column level (default)

`df.stack(future_stack=True)`  # Pandas 2.x recommended flag

`df.unstack(level=-1)`  # -1 = innermost index level (default)

#### Building a MultiIndex Dataframe

In [49]:
# Create a pivot with 2 index levels — this gives us a MultiIndex structure
df_multi = df.pivot_table(
    index=['Region', 'Month'],
    columns='Product',
    values='Units_Sold',
    aggfunc='sum'
)
df_multi.columns.name = None

print("Shape:", df_multi.shape)
print("Index type:", type(df_multi.index))
print("Index levels:", df_multi.index.names)
print()
df_multi.head(8)

Shape: (48, 4)
Index type: <class 'pandas.core.indexes.multi.MultiIndex'>
Index levels: ['Region', 'Month']



Laptop  Monitor  Phone  Tablet
Region Month                                
East   Apr       111      255     84     263
       Aug       227      193    148     208
       Dec       287      240    147     192
       Feb       248      153    213      57
       Jan       207      285    107     241
       Jul       252      147    118      87
       Jun       256      111    229     145
       Mar       173      120    114      64

#### Applying `stack()`

In [54]:
# stack() collapses the column headers into the innermost index level
df_stacked = df_multi.stack()
df_stacked.name = 'Units_Sold'   # give the Series a name

print("After stack():")
print("Type:", type(df_stacked))
print("Shape:", df_stacked.shape)
print("Index levels:", df_stacked.index.names)
print()
df_stacked.sample(12)

After stack():
Type: <class 'pandas.core.series.Series'>
Shape: (192,)
Index levels: ['Region', 'Month', None]



Region  Month         
North   Sep    Laptop     282
East    Oct    Monitor    241
North   Oct    Laptop     297
West    Oct    Phone       53
North   Nov    Laptop     270
        Sep    Monitor    201
South   Jan    Tablet     180
West    Oct    Laptop     246
South   Dec    Phone       62
East    Oct    Phone       79
West    Jun    Phone       88
South   Feb    Tablet     138
Name: Units_Sold, dtype: int64

#### Applying unstack() to reverse it

In [63]:
# unstack() lifts an index level back into columns
# By default lifts the innermost level (-1)

df_unstacked = df_stacked.unstack(level='Month')
df_unstacked.columns.name = None

df_unstacked = df_unstacked.reindex(month_order, axis= 1)

print("After unstack() — back to wide format:")
print("Shape:", df_unstacked.shape)
print()
df_unstacked.head(6)

After unstack() — back to wide format:
Shape: (16, 12)



Jan  Feb  Mar  Apr  May  Jun  Jul  Aug  Sep  Oct  Nov  Dec
Region                                                                    
East   Laptop   207  248  173  111  294  256  252  227  161   76  210  287
       Monitor  285  153  120  255  239  111  147  193  177  241  107  240
       Phone    107  213  114   84  133  229  118  148  142   79  272  147
       Tablet   241   57   64  263   77  145   87  208  266  275   74  192
North  Laptop   152  100   97  185  280  178   51  142  282  297  270  179
       Monitor  252  278  239  280  192   52  196  145  201  225   86  219

## Section 5: `crosstab()` -- Frequency & Contingency Table

`pd.crosstab()` counts how often combinations of values appear together.

Think of it as a quick pivot_table with aggfunc='count' — but with extra features.

**Syntax:**

`pd.crosstab(index, columns, values=None, aggfunc=None, normalize=False, margins=False)`

- Default: counts occurrences of each combination
- `normalize` → converts counts to proportions (row/column/overall %)
- `values + aggfunc` → aggregates a metric instead of counting

**Real-world use:** 
- Summarizing categorical breakdowns,
- survey analysis,
- checking data distribution across groups.

#### Basic Crosstab

In [68]:
# Add a 'Sales_Tier' column to make crosstab meaningful
df['Sales_Tier'] = pd.cut(
    df['Units_Sold'],
    bins=[0, 100, 200, 300],
    labels=['Low', 'Mid', 'High']
)

# How many records fall in each Region × Sales_Tier combination?
ct_basic = pd.crosstab(df['Region'], df['Sales_Tier'])
print("Record count by Region & Sales Tier\n")
ct_basic

Record count by Region & Sales Tier



Sales_Tier,Low,Mid,High
Region,,,
East,8,19,21
North,9,18,21
South,9,21,18
West,10,15,23


#### Crosstab with Normalize (proportions)

In [71]:
# normalize='index' → row-wise percentages (each region sums to 100%)
ct_pct = pd.crosstab(
    df['Region'],
    df['Sales_Tier'],
    normalize='index'
).round(3) * 100

print("% Distribution of Sales Tiers within each Region\n")
ct_pct

% Distribution of Sales Tiers within each Region



Sales_Tier,Low,Mid,High
Region,,,
East,16.7,39.6,43.8
North,18.8,37.5,43.8
South,18.8,43.8,37.5
West,20.8,31.2,47.9


#### Crosstab with Values + aggfunc

In [74]:
# Use crosstab like pivot_table — aggregate Revenue instead of counting
ct_revenue = pd.crosstab(
    index=df['Region'],
    columns=df['Product'],
    values=df['Revenue'],
    aggfunc='mean',
    margins=True,
    margins_name='Overall'
).round(0)

print("Average Revenue by Region & Product\n")
ct_revenue

Average Revenue by Region & Product



Product,Laptop,Monitor,Phone,Tablet,Overall
Region,,,,,
East,48111.0,39769.0,48941.0,54894.0,47929.0
North,45566.0,52645.0,43153.0,47135.0,47125.0
South,37522.0,46258.0,43418.0,35653.0,40712.0
West,40311.0,42450.0,55410.0,35659.0,43458.0
Overall,42878.0,45280.0,47730.0,43335.0,44806.0


## Section 6: `cut() / qcut()` -- Binning Continuous Data

Binning converts a continuous numeric column into discrete categories.

| Function | Basis | Use when... |
|----------|-------|-------------|
| `cut()`  | Fixed value boundaries | You know the meaningful thresholds |
| `qcut()` | Equal-frequency quantiles | You want equal-sized groups |

**Syntax:**

`pd.cut(series, bins=..., labels=..., right=True)`

`pd.qcut(series, q=..., labels=...)`        # q = number of quantiles

#### `cut()` with custom bins

In [79]:
# Bin Revenue into business-meaningful tiers
df['Revenue_Tier'] = pd.cut(
    df['Revenue'],
    bins=[0, 25000, 50000, 75000, 100000],
    labels=['Budget', 'Standard', 'Premium', 'Luxury'],
    right=True     # intervals are (left, right] — right boundary included
)

print("Value counts per Revenue Tier:")
print(df['Revenue_Tier'].value_counts().sort_index())
print()
df[['Revenue', 'Revenue_Tier']].sample(8, random_state=1)

Value counts per Revenue Tier:
Revenue_Tier
Budget      48
Standard    56
Premium     72
Luxury      16
Name: count, dtype: int64



,Revenue,Revenue_Tier
44,72592,Premium
69,32299,Standard
161,57202,Premium
35,65591,Premium
182,12368,Budget
11,35658,Standard
122,26646,Standard
81,16776,Budget


#### `qcut()` for Equal Frequency Bins

In [82]:
# qcut splits into equal-sized groups by frequency (not value range)
df['Units_Quartile'] = pd.qcut(
    df['Units_Sold'],
    q=4,
    labels=['Q1_Low', 'Q2_Mid', 'Q3_High', 'Q4_Top']
)

print("Records per Quartile (should be ~equal):")
print(df['Units_Quartile'].value_counts().sort_index())
print()

# Compare cut vs qcut bin boundaries
print("cut() bin ranges:  0–100 | 101–200 | 201–300  (fixed)")
print("qcut() bin ranges: data-driven, each has ~48 records")

Records per Quartile (should be ~equal):
Units_Quartile
Q1_Low     48
Q2_Mid     48
Q3_High    48
Q4_Top     48
Name: count, dtype: int64

cut() bin ranges:  0–100 | 101–200 | 201–300  (fixed)
qcut() bin ranges: data-driven, each has ~48 records


#### Combine Binning with Pivot Table

In [87]:
# Real-world pattern: bin → group → summarize
summary = df.pivot_table(
    index='Revenue_Tier',
    columns='Region',
    values='Units_Sold',
    aggfunc='mean',
    observed= False
).round(1)

summary.columns.name = None
print("Avg Units Sold by Revenue Tier & Region\n")
summary

Avg Units Sold by Revenue Tier & Region



,East,North,South,West
Revenue_Tier,,,,
Budget,195.2,169.9,165.6,149.7
Standard,172.6,192.0,181.8,198.2
Premium,185.6,200.0,136.2,185.2
Luxury,145.4,131.8,226.2,240.0


## Summary — Reshaping Cheatsheet

| Function       | Direction     | Aggregates? | Works on     |
|----------------|---------------|-------------|--------------|
| `pivot()`      | Long → Wide   | ❌ No        | Columns      |
| `pivot_table()`| Long → Wide   | ✅ Yes       | Columns      |
| `melt()`       | Wide → Long   | ❌ No        | Columns      |
| `stack()`      | Wide → Long   | ❌ No        | Index levels |
| `unstack()`    | Long → Wide   | ❌ No        | Index levels |
| `crosstab()`   | Summarize     | ✅ Optional  | Any columns  |
| `cut()`        | Bin by value  | —           | Numeric col  |
| `qcut()`       | Bin by freq   | —           | Numeric col  |

**Interview tip:**

When asked about reshaping — always clarify:
1. Do I need aggregation? → pivot_table / crosstab
2. Am I working with index levels? → stack / unstack
3. Do I need to normalize percentages? → crosstab(normalize=...)
4. Do I need to bin a continuous column? → cut / qcut